In [0]:
# Databricks Notebook 1: Data Generation

import numpy as np
import pandas as pd
from pyspark.sql import SparkSession

np.random.seed(42)

In [0]:
# Time range
weeks = pd.date_range(start="2023-01-01", periods=104, freq="W")

# Services in the city
services = ["metro", "bus", "private_transport", "water"]

# Base economics
base_prices = {
    "metro": 30,
    "bus": 10,
    "private_transport": 80,
    "water": 5
}

base_demand = {
    "metro": 500000,
    "bus": 800000,
    "private_transport": 600000,
    "water": 1e6
}

# Price elasticity
elasticity = {
    "metro": -1.3,
    "bus": -1.1,
    "private_transport": -0.8,
    "water": -0.4
}

In [0]:
def seasonality(t):
    return 1 + 0.1 * np.sin(2 * np.pi * t / 52)

def noise():
    return np.random.normal(0, 0.05)

def festival_flag(week):
    # Pongal, Diwali, year-end
    return 1 if week.month in [1, 10, 11, 12] else 0

def holiday_flag(week):
    return 1 if week.weekday() >= 5 else 0

In [0]:
rows = []

base_population = 1.1e7

for t, week in enumerate(weeks):

    # Temporal signals
    festival = festival_flag(week)
    holiday = holiday_flag(week)

    # Population drift (migration + festivals)
    pop_drift = 1 + np.random.uniform(-0.05, 0.1)
    if festival:
        pop_drift += 0.1

    population = base_population * pop_drift

    # Infrastructure dynamics
    congestion = 0.65 + np.random.uniform(-0.1, 0.1)
    congestion = min(1, max(0, congestion))

    signal_load = congestion * 0.6

    for s in services:

        # Price fluctuation
        price = base_prices[s] * (1 + np.random.uniform(-0.1, 0.1))

        # Demand via elasticity
        demand = base_demand[s] * ((price / base_prices[s]) ** elasticity[s])

        # Apply seasonality + noise
        demand *= seasonality(t)
        demand *= (1 + noise())

        # Traffic effects
        if s == "metro":
            demand *= (1 + congestion)
        elif s == "private_transport":
            demand *= (1 - congestion)

        # Festival / holiday behavior
        if festival:
            demand *= 1.2

        if holiday:
            if s in ["metro", "bus"]:
                demand *= 0.9
            if s == "private_transport":
                demand *= 1.1

        demand = max(1, demand)

        revenue = demand * price

        rows.append({
            "week": str(week),
            "service": s,
            "price": float(price),
            "demand": float(demand),
            "revenue": float(revenue),
            "population": float(population),
            "congestion": float(congestion),
            "signal_load": float(signal_load),
            "festival": int(festival),
            "holiday": int(holiday)
        })

df = pd.DataFrame(rows)

print("Sample Data:")
print(df.head())

print("\nShape:", df.shape)

Sample Data:
                  week            service  ...  festival  holiday
0  2023-01-01 00:00:00              metro  ...         1        1
1  2023-01-01 00:00:00                bus  ...         1        1
2  2023-01-01 00:00:00  private_transport  ...         1        1
3  2023-01-01 00:00:00              water  ...         1        1
4  2023-01-08 00:00:00              metro  ...         1        1

[5 rows x 10 columns]

Shape: (416, 10)


In [0]:
spark_df = spark.createDataFrame(df)

spark_df.printSchema()

root
 |-- week: string (nullable = true)
 |-- service: string (nullable = true)
 |-- price: double (nullable = true)
 |-- demand: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- population: double (nullable = true)
 |-- congestion: double (nullable = true)
 |-- signal_load: double (nullable = true)
 |-- festival: long (nullable = true)
 |-- holiday: long (nullable = true)



In [0]:
# Drop table if exists (clean rerun)
spark.sql("DROP TABLE IF EXISTS city_data")

# Save as managed Delta table (NO DBFS PATH)
spark_df.write.format("delta").mode("overwrite").saveAsTable("city_data")

In [0]:
# Quick check
spark.sql("SELECT * FROM city_data LIMIT 10").show()

# Count rows
spark.sql("SELECT COUNT(*) FROM city_data").show()

+-------------------+-----------------+------------------+------------------+--------------------+--------------------+------------------+-------------------+--------+-------+
|               week|          service|             price|            demand|             revenue|          population|        congestion|        signal_load|festival|holiday|
+-------------------+-----------------+------------------+------------------+--------------------+--------------------+------------------+-------------------+--------+-------+
|2023-01-01 00:00:00|            metro| 31.39196365086843| 836625.3166004848| 2.626331152811871E7| 1.216799119609815E7|0.7401428612819833|0.44408571676918995|       1|      1|
|2023-01-01 00:00:00|              bus| 9.311989040672405|  949373.892656423|    8840559.28391711| 1.216799119609815E7|0.7401428612819833|0.44408571676918995|       1|      1|
|2023-01-01 00:00:00|private_transport|  72.9293377946912| 224712.6032937637| 1.638814135233533E7| 1.216799119609815E7|0